# Preprocesamiento de Datos: Determinantes de Salud Mental en Bogotá

**DataJam Edición 3 — 2026** | Equipo GEMMA 2.0

Este notebook realiza la carga, limpieza e integración de cuatro fuentes públicas
de datos (conducta suicida, encuesta de percepción, infraestructura verde y
límites de localidades) para construir el dataset maestro geoespacial que
alimenta el análisis del Notebook 2 y el dashboard interactivo del proyecto.

**Objetivo del preprocesamiento:** producir un conjunto de datos único,
armonizado a nivel de localidad, libre de inconsistencias de nomenclatura entre
fuentes, listo para análisis estadístico y geoespacial.

**Salida principal:** `master_mental_health_bogota_2025.geojson`

## **Fuentes de datos utilizadas**
Incluyendo la descarga de soportes adicionales (diccionarios de metadatos y relación variable significado).

| Fuente | Descripción | URL |
|---|---|---|
|Conducta suicida Bogotá (OSB) | Casos de ideación, intento y suicidio consumado 2023-2025| https://datosabiertos.bogota.gov.co/dataset/tasa-de-suicidio-en-bogota-d-c |
| Encuesta Distrital de Percepción 2025 | Determinantes sociales de salud percibida | https://www.sdp.gov.co/gestion-estudios-estrategicos/informacion-estadisticas/encuesta-distrital-percepcion |
| Sistema Distrital de Parques y Escenarios Públicos<br>Indicador Espacio Público Ciudad. Bogotá D.C. | Inventario de parques y espacio público por localidad | https://datosabiertos.bogota.gov.co/dataset/sistema-distrital-de-parques-y-escenarios-publicos-deportivos<br>https://datosabiertos.bogota.gov.co/dataset/indicador-espacio-publico-ciudad-bogota-d-c  |
| Limites Político-Administrativos (UPL y Localidades) | Límites geográficos oficiales de las localidades de Bogotá | https://datosabiertos.bogota.gov.co/dataset/localidad-bogota-d-c |
|Población en Bogotá D.C. 2005-2035|Población por cada localidad de Bogotá|https://datosabiertos.bogota.gov.co/dataset/piramide-poblacional-bogota-d-c

## **Preprocesamiento de los datos primarios**

### Estructura de datos crudos esperada

```
data/raw/
  01/
    conducta_suicida_bogota.csv
    metadato_osb_salud_mental_ideacion_e_intento.csv
    metadato_osb_salud_mental_suicidio_consumado.csv
    suicidio_consumado_bogota.csv
  02/
    diccionario_base.xlsx
    encuesta_percepcion_2025.csv
  03/
    indicador_espacio_publico_dadep.geojson
    sistema_parques_bogota.geojson
  04/
    localidades_bogota.geojson
  05/
    demografia_poblacion_localidad.csv
    metadato_osb_demografia_poblacion.csv
```


### 1. Instalación de dependencias.

In [ ]:
import os
import shutil
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from google.colab import drive

warnings.filterwarnings("ignore")

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


### 1.1 Inicialización de las rutas del repositorio

In [ ]:
PROJECT_NAME = "prueba" # Indique el nombre de la carpeta donde se migrará los archivos que se clonarán del repositorio

REPO_URL = "https://github.com/ILuuI/datajam-2026-mental-health-bogota.git"
BRANCH = "main"
PROJECT_PATH = f"/content/drive/MyDrive/{PROJECT_NAME}/"
TMP_CLONE_PATH = "/content/repo_min"

NOTEBOOKS_PATH = os.path.join(PROJECT_PATH, "notebooks")
RAW_ZIPS_PATH = os.path.join(PROJECT_PATH, "data", "raw_zips")

### 1.2 Creación de funciones auxiliares (etapa de clonación y verificación de archivos)

In [ ]:
def folder_has_files(path):
    """
    Retorna True si la carpeta existe y contiene al menos un archivo.
    """
    return os.path.isdir(path) and len(os.listdir(path)) > 0


def clone_minimum(target_subfolders):
    """
    Clona el repositorio con sparse-checkout y copia SOLO las subcarpetas
    indicadas (lista de tuplas: (ruta_relativa_en_repo, ruta_destino_en_drive)).
    """
    if os.path.exists(TMP_CLONE_PATH):
        shutil.rmtree(TMP_CLONE_PATH)

    sparse_targets = " ".join(rel for rel, _ in target_subfolders)
    os.system(
        f"git clone --filter=blob:none --no-checkout --depth 1 -b {BRANCH} "
        f"{REPO_URL} {TMP_CLONE_PATH}"
    )
    os.system(
        f"cd {TMP_CLONE_PATH} && "
        f"git sparse-checkout init --cone && "
        f"git sparse-checkout set {sparse_targets} && "
        f"git checkout"
    )

    for rel, dest in target_subfolders:
        source = os.path.join(TMP_CLONE_PATH, rel)
        os.makedirs(os.path.dirname(dest) if not dest.endswith(os.sep) else dest, exist_ok=True)
        shutil.copytree(source, dest, dirs_exist_ok=True)

    shutil.rmtree(TMP_CLONE_PATH)

In [ ]:
# Verificación de existencia previa
if os.path.exists(PROJECT_PATH):
    print(f"    La carpeta '{PROJECT_NAME}' ya existe en tu Drive:")
    print(f"    {PROJECT_PATH}")
    answer = input(
        "¿Deseas sobrescribirla y volver a descargar todo desde cero? (s/n): "
    ).strip().lower()

    if answer == "s":
        shutil.rmtree(PROJECT_PATH)
        os.makedirs(PROJECT_PATH, exist_ok=True)
        print(f"Carpeta reiniciada: {PROJECT_PATH}")
        clone_minimum([
            ("notebooks", NOTEBOOKS_PATH),
            ("data/raw_zips", RAW_ZIPS_PATH),
        ])
        os.makedirs(os.path.join(PROJECT_PATH, "data", "raw"), exist_ok=True)

    else:
        missing = []
        if not folder_has_files(NOTEBOOKS_PATH):
            missing.append(("notebooks", NOTEBOOKS_PATH))
        if not folder_has_files(RAW_ZIPS_PATH):
            missing.append(("data/raw_zips", RAW_ZIPS_PATH))

        if missing:
            print("Se conserva la carpeta existente, pero se detectaron elementos faltantes:")
            for rel, dest in missing:
                print(f"   - Falta o está vacía: {dest}")
            print("Descargando únicamente lo faltante (sin tocar el resto)...")
            clone_minimum(missing)
            os.makedirs(os.path.join(PROJECT_PATH, "data", "raw"), exist_ok=True)
            print("Elementos faltantes descargados.")
        else:
            print("Todo en orden: 'notebooks' y 'data/raw_zips' ya están completos. No se realizaron cambios.")

else:
    os.makedirs(PROJECT_PATH, exist_ok=True)
    clone_minimum([
        ("notebooks", NOTEBOOKS_PATH),
        ("data/raw_zips", RAW_ZIPS_PATH),
    ])
    os.makedirs(os.path.join(PROJECT_PATH, "data", "raw"), exist_ok=True)

### 1.3 Verificación final del estado del proyecto

In [ ]:
print(f"\nEstado final del proyecto en: {PROJECT_PATH}")
for root, dirs, files in os.walk(PROJECT_PATH):
    level = root.replace(str(PROJECT_PATH), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


Estado final del proyecto en: /content/drive/MyDrive/prueba/
/
notebooks/
  001_requirements.txt
  001_pre_processing.ipynb
  002_post_processing_and_analysis.ipynb
  002_requirements.txt
data/
  raw_zips/
    raw.rar
  raw/


### 1.4 Inicialización de las rutas del espacio de trabajo

In [ ]:
RAW_DATA_PATH = os.path.join(PROJECT_PATH, "data/raw/")
PROCESSED_DATA_PATH = os.path.join(PROJECT_PATH, "data/processed/")
OUTPUTS_FIG_PATH = os.path.join(PROJECT_PATH, "outputs/figures/")
OUTPUTS_TABLE_PATH = os.path.join(PROJECT_PATH, "outputs/tables/")

RAW_ZIP_PATH = f"{PROJECT_PATH}data/raw_zips/raw.rar"
DESTINATION = os.path.join(PROJECT_PATH, "data/raw")

for path in [
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    OUTPUTS_FIG_PATH,
    OUTPUTS_TABLE_PATH,
]:
    os.makedirs(path, exist_ok=True)

print(f"Inicialización del Entorno 001 completada.")

Inicialización del Entorno 001 completada.


### 1.5 Extracción del conjunto de datos espaciales y tabulares (.csv, .geojson).

In [ ]:
!unrar x $RAW_ZIP_PATH $DESTINATION


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/prueba/data/raw_zips/raw.rar

Creating    /content/drive/MyDrive/prueba/data/raw/02                 OK
Extracting  /content/drive/MyDrive/prueba/data/raw/02/diccionario_base.xlsx       0%  OK 
Extracting  /content/drive/MyDrive/prueba/data/raw/02/encuesta_percepcion_2025.csv      15%  OK 
Creating    /content/drive/MyDrive/prueba/data/raw/03                 OK
Extracting  /content/drive/MyDrive/prueba/data/raw/03/indicador_espacio_publico_dadep.geojson      19%  OK 
Extracting  /content/drive/MyDrive/prueba/data/raw/03/sistema_parques_bogota.geojson      63%  OK 
Creating    /content/drive/MyDrive/prueba/data/raw/04                 OK
Extracting  /content/drive/MyDrive/prueba/data/raw/04/localidades_bogota.geojson      69%  OK 
Creating    /content/drive/MyDrive/prueba/data/raw/05                 OK
Extracting  /content/drive/My

### 1.6 Función de importación de archivos
Creación de función auxiliar para detectar de forma automática la codificación y el delimitador de cada archivo de extensión ".csv".

In [ ]:
def load_district_csv(filepath: str) -> pd.DataFrame:
    """Carga archivos CSV comprobando las codificaciones y los delimitadores estándar de los datos públicos colombianos."""
    encodings = ["latin1", "utf-8-sig", "utf-8", "cp1252"]
    delimiters = [";", ","]

    for enc in encodings:
        for sep in delimiters:
            try:
                df = pd.read_csv(
                    filepath,
                    sep=sep,
                    encoding=enc,
                    low_memory=False,
                    on_bad_lines="skip",
                )
                if df.shape[1] > 1:
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue

    return pd.read_csv(
        filepath,
        sep=None,
        engine="python",
        encoding="latin1",
        on_bad_lines="skip",
    )

### 2. Carga y exploración de fuentes primarias

Los archivos originales fueron renombrados con nombres descriptivos y
consistentes (por ejemplo, `conducta_suicida_bogota.csv`,
`encuesta_percepcion_2025.csv`), en lugar de conservar los nombres genéricos
o poco claros con los que se descargaron del portal de origen. Esto facilita
la lectura del código y la referencia a cada fuente a lo largo del notebook.
La estructura de carpetas y nombres de archivo resultante se muestra en la
sección anterior.

In [ ]:
# 2.1 Indicador y Eventos de Conducta Suicida (SaluData / SDS)
suicidal_behavior_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "01/conducta_suicida_bogota.csv")
)

completed_suicide_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "01/suicidio_consumado_bogota.csv")
)

# 2.2 Encuesta Distrital de Percepción 2025 (SDP)
perception_survey_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "02/encuesta_percepcion_2025.csv")
)

# 2.3 Capas de Zonas Verdes y Espacio Público
# 2.3.1 Indicador Espacio Público Ciudad (DADEP)
public_space_indicator_gdf = gpd.read_file(
    os.path.join(RAW_DATA_PATH, "03/indicador_espacio_publico_dadep.geojson")
).to_crs(epsg=4326)

# 2.3.2 Sistema Distrital de Parques y Escenarios Públicos Deportivos (IDRD)
parks_system_gdf = gpd.read_file(
    os.path.join(RAW_DATA_PATH, "03/sistema_parques_bogota.geojson")
).to_crs(epsg=4326)

# 2.4 Límites Político-Administrativos (Localidades / IDECA - SDP)
localities_gdf = gpd.read_file(
    os.path.join(RAW_DATA_PATH, "04/localidades_bogota.geojson")
).to_crs(epsg=4326)

# 2.5 Población en Bogotá D.C. 2005-2035 (Localidades)
population_raw_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "05/demografia_poblacion_localidad.csv")
)

print("Conjuntos de datos primarios correctamente cargados para etapa de preprocesamiento.")

Conjuntos de datos primarios correctamente cargados para etapa de preprocesamiento.


### 2.1 Funciones auxiliares de limpieza de texto: corrección de codificación y normalización de identificadores

Se definen dos funciones reutilizables a lo largo del notebook: la primera
corrige errores de codificación (mojibake) presentes en varios archivos
crudos, y la segunda estandariza los identificadores de texto (mayúsculas,
sin tildes) y armoniza casos particulares de nomenclatura de localidades
detectados entre fuentes (por ejemplo, "CANDELARIA" vs. "LA CANDELARIA").

In [ ]:
# 2.1.1. Función maestra para corregir problemas de codificación (Mojibake)
def fix_mojibake_encoding(text: str) -> str:
    if pd.isna(text):
        return text
    # Parte de los carácteres que se observan fueron los identificados en los distintos conjuntos de datos
    replacements = {
        "Ã±": "ñ", "Ã‘": "Ñ", "Ã¡": "á", "Ã©": "é", "Ã": "í",
        "Ã³": "ó", "Ãº": "ú", "Ã": "Á", "Ã‰": "É", "Ã": "Í",
        "Ã“": "Ó", "Ãš": "Ú", "¥": "Ñ", "Í\xad": "í", "Í³": "ó"
    }
    text_str = str(text)
    for bad_char, good_char in replacements.items():
        text_str = text_str.replace(bad_char, good_char)
    return text_str.strip()

# 2.1.2 Función para estandarizar identificadores (mayúsculas y sin tildes)
def clean_text_identifier(text: str) -> str:
    if pd.isna(text):
        return np.nan

    # Reparación de la codificación previa a la estandarización
    text = fix_mojibake_encoding(text)
    text = str(text).upper().strip()
    accent_map = {
        "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U", "Ñ": "N",
    }
    for accented, standard in accent_map.items():
        text = text.replace(accented, standard)
    if text == "CANDELARIA":
        text = "LA CANDELARIA"
    return text

def fix_park_text(text: str) -> str:
    if pd.isna(text):
        return text
    text_str = str(text).replace("¥", "Ñ").replace("Ã±", "Ñ")
    return clean_text_identifier(text_str)

In [ ]:
# Limpieza estructural de los DataFrames tabulares
for df in [completed_suicide_df, perception_survey_df, suicidal_behavior_df, population_raw_df]:
    # Normalizar encabezados (quitar BOM ï»¿, espacios extra y pasar a mayúsculas)
    df.columns = df.columns.str.replace("ï»¿", "").str.strip().str.upper()

    # Corregir textos en todas las columnas categóricas desde el origen
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(fix_mojibake_encoding)

print("Encabezados normalizados y codificación de caracteres reparada en los DataFrames primarios.")

Encabezados normalizados y codificación de caracteres reparada en los DataFrames primarios.


#### Verificación de nombres de localidad en el dataset de población

Se listan los códigos y nombres únicos de localidad presentes en el dataset de demografía, con el fin de identificar de antemano si esta fuente utiliza alguna convención de nomenclatura distinta a las ya armonizadas.

In [ ]:
population_raw_df[["CODIGO_LOCALIDAD", "NOMBRE_LOCALIDAD"]].drop_duplicates().sort_values("CODIGO_LOCALIDAD")

,CODIGO_LOCALIDAD,NOMBRE_LOCALIDAD
0,0,Bogotá
6,1,Usaquén
12,2,Chapinero
18,3,Santa Fe
24,4,San Cristóbal
30,5,Usme
36,6,Tunjuelito
42,7,Bosa
48,8,Kennedy
54,9,Fontibón


### 2.3 Normalización de claves geográficas

Se estandarizan las claves de localidad de la capa geoespacial oficial: se detecta automáticamente la columna de nombre y de código de localidad, se aplica la función `clean_text_identifier` al nombre para generar
`locality_clean`, y se extrae el código numérico de 2 dígitos como `locality_code`. Estas dos claves servirán como llave de integración con el resto de las fuentes (conducta suicida, encuesta de percepción, parques y
demografía) más adelante en el notebook.

In [ ]:
loc_name_col = [
    c
    for c in localities_gdf.columns
    if "NOM" in c.upper() or "LOC" in c.upper()
][0]
loc_code_col = [c for c in localities_gdf.columns if "COD" in c.upper()][0]

localities_gdf["locality_clean"] = localities_gdf[loc_name_col].apply(
    clean_text_identifier
)
localities_gdf["locality_code"] = (
    localities_gdf[loc_code_col].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)
)

print(f"Claves geográficas (nombre y código de 2 dígitos) estandarizadas en localidades.")

Claves geográficas (nombre y código de 2 dígitos) estandarizadas en localidades.


In [ ]:
# 2.3.1 Agregación de población promedio 2023-2025 por localidad
# Se excluye el código 0 (total Bogotá) para evitar duplicar la suma de las 20 localidades
population_localities_df = population_raw_df[population_raw_df["CODIGO_LOCALIDAD"] != 0].copy()

# Filtro de la ventana de análisis (2023-2025)
population_localities_df = population_localities_df[
    population_localities_df["ANO"].between(2023, 2025)
]

# Suma de población total por localidad y año (colapsando sexo, edad, curso de vida, grupo de edad)
population_by_year_df = (
    population_localities_df.groupby(["NOMBRE_LOCALIDAD", "ANO"])["POBLACION"]
    .sum()
    .reset_index()
)

# Promedio de los 3 años (2023, 2024, 2025) por localidad
population_avg_df = (
    population_by_year_df.groupby("NOMBRE_LOCALIDAD")["POBLACION"]
    .mean()
    .reset_index()
    .rename(columns={"POBLACION": "avg_population_2023_2025"})
)

# Estandarización del nombre de localidad con la misma función usada en el resto del pipeline
population_avg_df["locality_clean"] = population_avg_df["NOMBRE_LOCALIDAD"].apply(
    clean_text_identifier
)

print(f"Población promedio 2023-2025 calculada para {population_avg_df.shape[0]} localidades.")
population_avg_df[["locality_clean", "avg_population_2023_2025"]]

Población promedio 2023-2025 calculada para 20 localidades.


,locality_clean,avg_population_2023_2025
0,ANTONIO NARINO,7.797167e+04
1,BARRIOS UNIDOS,1.357047e+05
2,BOSA,7.602703e+05
3,CHAPINERO,1.616237e+05
4,CIUDAD BOLIVAR,6.773487e+05
5,ENGATIVA,8.310153e+05
6,FONTIBON,3.854040e+05
7,KENNEDY,1.099031e+06
8,LA CANDELARIA,1.668167e+04
9,LOS MARTIRES,7.538400e+04


In [ ]:
# 2.3.2 Validación cruzada de nombres de localidad entre fuentes
localities_master = set(localities_gdf["locality_clean"])
localities_population = set(population_avg_df["locality_clean"])

only_in_master = localities_master - localities_population
only_in_population = localities_population - localities_master

print(f"Localidades en localities_gdf: {len(localities_master)}")
print(f"Localidades en population_avg_df: {len(localities_population)}")

if only_in_master:
    print("\nPresentes en localities_gdf pero NO en population_avg_df (no encontrarán match en el merge):")
    for loc in sorted(only_in_master):
        print(f"   - {loc}")

if only_in_population:
    print("\nPresentes en population_avg_df pero NO en localities_gdf (sobrarán, no se usarán):")
    for loc in sorted(only_in_population):
        print(f"   - {loc}")

if not only_in_master and not only_in_population:
    print("\nLas 20 localidades coinciden exactamente entre ambas fuentes.")

Localidades en localities_gdf: 20
Localidades en population_avg_df: 20

Las 20 localidades coinciden exactamente entre ambas fuentes.


### 3. Cálculo de indicadores epidemiológicos

A partir de los microdatos de conducta suicida, se identifican dinámicamente las columnas de localidad, sexo y año, se extrae y estandariza el código de localidad para vincularlo al nombre oficial (`locality_clean`) mediante el mapeo construido en la sección anterior, y se filtra la ventana de análisis
al periodo 2023-2025. Finalmente, se consolida una tabla epidemiológica con el total de eventos y su desagregación por sexo, por localidad.

In [ ]:
suicide_df = completed_suicide_df.copy()

# 3.1 Identificación de columnas de localidad, sexo y año en los microdatos
loc_col = [
    c
    for c in suicide_df.columns
    if "LOC" in c.upper() or "MUNICIPIO" in c.upper()
][0]
sex_col = [c for c in suicide_df.columns if "SEX" in c.upper()][0]
year_col = [c for c in suicide_df.columns if "ANO" in c.upper()][0]

# 3.2 Extracción del código numérico de 2 dígitos de la localidad para posterior estandarización
suicide_df["locality_code"] = (
    suicide_df[loc_col].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)
)

# 3.3 Mapeo del código numérico al nombre oficial de la localidad usando localities_gdf
loc_mapping = dict(
    zip(localities_gdf["locality_code"], localities_gdf["locality_clean"])
)
suicide_df["locality_clean"] = suicide_df["locality_code"].map(loc_mapping)

# 3.4 Filtro de la ventana de análisis (2023-2025), consistente con el resto del estudio
suicide_df["year"] = pd.to_numeric(suicide_df[year_col], errors="coerce")
suicide_df = suicide_df[suicide_df["year"].between(2023, 2025)]

# Aplicación de filtro únicamente a las 20 localidades oficiales válidas de Bogotá
suicide_clean_df = suicide_df[suicide_df["locality_clean"].notna()].copy()

# 3.5 Conteo total de eventos por localidad (ya restringido a 2023-2025)
suicide_summary_df = (
    suicide_clean_df.groupby("locality_clean")
    .size()
    .reset_index(name="historico_total_suicidios_localidad")
)

# 3.6 Tabulación cruzada por Sexo
sex_pivot_df = (
    pd.crosstab(
        suicide_clean_df["locality_clean"], suicide_clean_df[sex_col]
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Hombre": "hombres",
        "Mujer": "mujeres"
    })
)

# 3.7 Consolidación de la tabla epidemiológica final
epidemiological_summary_df = suicide_summary_df.merge(
    sex_pivot_df, on="locality_clean", how="left"
)

print(f"Indicadores epidemiológicos consolidados exitosamente para {len(epidemiological_summary_df)} localidades oficiales (ventana 2023-2025).")
epidemiological_summary_df.head(20)

Indicadores epidemiológicos consolidados exitosamente para 19 localidades oficiales (ventana 2023-2025).


,locality_clean,historico_total_suicidios_localidad,hombres,mujeres
0,ANTONIO NARINO,12,7,5
1,BARRIOS UNIDOS,21,16,5
2,BOSA,107,78,29
3,CHAPINERO,43,35,8
4,CIUDAD BOLIVAR,113,85,28
5,ENGATIVA,100,76,24
6,FONTIBON,41,30,11
7,KENNEDY,139,96,43
8,LA CANDELARIA,5,3,2
9,LOS MARTIRES,10,6,4


### 4. Resultados agregados de la Encuesta de Percepción 2025 (SDP)

Se estandarizan los encabezados de los microdatos de la Encuesta Distrital de Percepción, se identifica la columna de localidad y se le aplica la misma función de normalización usada en las secciones anteriores, para asegurar consistencia con la clave `locality_clean` del resto del pipeline.

Finalmente, se calcula el tamaño de la muestra encuestada por localidad, como indicador de representatividad.

In [ ]:
# 4.1 Estandarización de los encabezados de las columnas
perception_survey_df.columns = [
    c.upper().strip() for c in perception_survey_df.columns
]

# 4.2 Identificación de la columna "localidad" en los microdatos de la encuesta de percepción
survey_loc_col = [
    c
    for c in perception_survey_df.columns
    if "LOC" in c or "LOCALIDAD" in c or "COD_LOC" in c
][0]
perception_survey_df["locality_clean"] = perception_survey_df[
    survey_loc_col
].apply(clean_text_identifier)

# 4.3 Indicadores agregados de la encuesta por "localidad"
survey_summary_df = (
    perception_survey_df.groupby("locality_clean")
    .agg(total_surveyed_sample=(survey_loc_col, "count"))
    .reset_index()
)

print("Se han agregado los indicadores de la encuesta de percepción por 'localidad'.")
survey_summary_df.head()

Se han agregado los indicadores de la encuesta de percepción por 'localidad'.


,locality_clean,total_surveyed_sample
0,1,834
1,10,1232
2,11,1802
3,12,270
4,13,335


### 5. Análisis espacial: espacios verdes por localidad

Se normalizan los nombres de localidad del sistema de parques, incluyendo la corrección explícita de abreviaturas propias de esta fuente ("MARTIRES",
"RAFAEL URIBE", "SANTAFE") que no coinciden con los nombres oficiales completos usados en el resto de fuentes. Se calcula el área de cada parque (a partir del atributo de área si está disponible, o mediante proyección geográfica si no lo está), y se agregan el número de parques y el área verde total por localidad.

In [ ]:
# 5.1 Crear copia del dataset de parques y normalizar nombres
parks_clean_df = parks_system_gdf.copy()
parks_clean_df["locality_clean"] = parks_clean_df["LocNombre"].apply(
    fix_park_text
)
locality_name_fixes = { # Correción de problematica entre nombres de la misma localidad en distintas base de datos
    "MARTIRES": "LOS MARTIRES",
    "RAFAEL URIBE": "RAFAEL URIBE URIBE",
    "SANTAFE": "SANTA FE",
}
parks_clean_df["locality_clean"] = parks_clean_df["locality_clean"].replace(locality_name_fixes)

# 5.2 Convertir área a formato numérico
if "SHAPE_Area" in parks_clean_df.columns:
    parks_clean_df["area_sq_meters"] = pd.to_numeric(
        parks_clean_df["SHAPE_Area"], errors="coerce"
    ).fillna(0)
else:
    parks_projected = parks_clean_df.to_crs(epsg=9377)
    parks_clean_df["area_sq_meters"] = parks_projected.geometry.area

# 5.3 Agrupar métricas por las localidades oficiales
parks_summary_df = (
    parks_clean_df.groupby("locality_clean")
    .agg(
        parks_count=("area_sq_meters", "count"),
        total_green_area_sqm=("area_sq_meters", "sum"),
    )
    .reset_index()
)

print(f"Métricas espaciales consolidadas para {len(parks_summary_df)} localidades de Bogotá.")
parks_summary_df.head(20)

Métricas espaciales consolidadas para 20 localidades de Bogotá.


,locality_clean,parks_count,total_green_area_sqm
0,ANTONIO NARINO,55,2.901663e+05
1,BARRIOS UNIDOS,123,1.742276e+06
2,BOSA,251,1.222724e+06
3,CHAPINERO,160,7.952233e+05
4,CIUDAD BOLIVAR,447,2.212691e+06
5,ENGATIVA,552,6.179346e+06
6,FONTIBON,280,1.609459e+06
7,KENNEDY,552,3.792798e+06
8,LA CANDELARIA,10,1.946796e+04
9,LOS MARTIRES,47,2.032605e+05


### 6. Integración y Exportación de los Datasets Principales.
Se construye el GeoDataFrame maestro mediante uniones sucesivas (`left join`) entre la capa geoespacial de localidades y los cuatro indicadores agregados:
infraestructura verde, encuesta de percepción, demografía y conducta suicida.

Se rellenan explícitamente con cero los valores nulos de parques y área verde (ausencia real de infraestructura catalogada) y de eventos de conducta suicida (localidades sin casos registrados), distinguiendo estos ceros legítimos de los que resultarían de un error de cruce por nombres no armonizados.

In [ ]:
# 6.1 Fusión de capas territoriales en el mapa maestro
master_gdf = localities_gdf[["locality_clean", "geometry"]].merge(
    parks_summary_df, on="locality_clean", how="left"
)
master_gdf = master_gdf.merge(
    survey_summary_df, on="locality_clean", how="left"
)
master_gdf = master_gdf.merge(
    population_avg_df[["locality_clean", "avg_population_2023_2025"]],
    on="locality_clean",
    how="left",
)
master_gdf = master_gdf.merge(
    epidemiological_summary_df, on="locality_clean", how="left"
)

master_gdf["parks_count"] = master_gdf["parks_count"].fillna(0)
master_gdf["total_green_area_sqm"] = master_gdf["total_green_area_sqm"].fillna(0)

# Nota: Las localidades sin eventos registrados representan 0 casos reales, por lo que sí es correcto rellenar con 0
suicide_count_cols = ["historico_total_suicidios_localidad", "hombres", "mujeres"]
for col in suicide_count_cols:
    if col in master_gdf.columns:
        master_gdf[col] = master_gdf[col].fillna(0)

# Estandarización de nombres de columna a la convención usada en el resto del pipeline y el dashboard
master_gdf = master_gdf.rename(columns={
    "historico_total_suicidios_localidad": "total_suicides_2023_2025",
    "hombres": "hombre",
    "mujeres": "mujer",
})

In [ ]:
# 6.2 Comprobación de la unión de población y suicidios en master_gdf
print("Columnas actuales de master_gdf:")
print(master_gdf.columns.tolist())

cols_check = [
    "locality_clean", "parks_count", "total_green_area_sqm",
    "avg_population_2023_2025", "historico_total_suicidios_localidad",
    "hombres", "mujeres"
]
cols_check = [c for c in cols_check if c in master_gdf.columns]

print("\nValores nulos por columna clave:")
print(master_gdf[cols_check].isna().sum())

print("\nVista previa (sin geometría):")
master_gdf.drop(columns="geometry")[cols_check]

Columnas actuales de master_gdf:
['locality_clean', 'geometry', 'parks_count', 'total_green_area_sqm', 'total_surveyed_sample', 'avg_population_2023_2025', 'total_suicides_2023_2025', 'hombre', 'mujer']

Valores nulos por columna clave:
locality_clean              0
parks_count                 0
total_green_area_sqm        0
avg_population_2023_2025    0
dtype: int64

Vista previa (sin geometría):


,locality_clean,parks_count,total_green_area_sqm,avg_population_2023_2025
0,ANTONIO NARINO,55,2.901663e+05,7.797167e+04
1,TUNJUELITO,54,8.794631e+05,1.756830e+05
2,RAFAEL URIBE URIBE,262,1.112265e+06,3.754383e+05
3,LA CANDELARIA,10,1.946796e+04,1.668167e+04
4,BARRIOS UNIDOS,123,1.742276e+06,1.357047e+05
5,TEUSAQUILLO,134,2.023925e+06,1.548837e+05
6,PUENTE ARANDA,269,1.076713e+06,2.496377e+05
7,LOS MARTIRES,47,2.032605e+05,7.538400e+04
8,SUMAPAZ,5,9.259314e+03,3.300667e+03
9,USAQUEN,472,3.194473e+06,5.859093e+05


In [ ]:
# 6.5 Exportación de todos los archivos procesados a data/processed/
master_gdf.to_file(
    os.path.join(
        PROCESSED_DATA_PATH, "master_mental_health_bogota_2025.geojson"
    ),
    driver="GeoJSON",
)
master_gdf.drop(columns="geometry").to_csv(
    os.path.join(PROCESSED_DATA_PATH, "master_mental_health_bogota_2025.csv"),
    index=False,
)

### 7. Preprocesamiento final: microdatos de conducta suicida y encuesta de percepción

In [ ]:
# 7.1 Estandarización de localidad por código de 2 dígitos
loc_col_final = [c for c in completed_suicide_df.columns if "LOC" in c.upper() or "MUNICIPIO" in c.upper()][0]
completed_suicide_df["temp_code"] = completed_suicide_df[loc_col_final].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)

loc_mapping = dict(zip(localities_gdf["locality_code"], localities_gdf["locality_clean"]))
completed_suicide_df[loc_col_final] = completed_suicide_df["temp_code"].map(loc_mapping)

if "ANO_DEL_HECHO" in completed_suicide_df.columns:
    completed_suicide_df["year"] = pd.to_numeric(completed_suicide_df["ANO_DEL_HECHO"], errors="coerce")

# Filtro a las 20 localidades oficiales (códigos 01-20)
valid_localities = [str(i).zfill(2) for i in range(1, 21)]
suicide_final_export = completed_suicide_df[completed_suicide_df["temp_code"].isin(valid_localities)].drop(columns=["temp_code"])

suicide_final_export.to_csv(
    os.path.join(PROCESSED_DATA_PATH, "suicide_events_clean.csv"),
    index=False,
    encoding="utf-8-sig",
)

# 7.2 Conversión a numérico y homologación de códigos de No sabe/No responde (>=90) a NaN
model_vars = ["A3", "A4", "A5", "A6X2"]
for col in model_vars:
    if col in perception_survey_df.columns:
        perception_survey_df[col] = pd.to_numeric(perception_survey_df[col], errors="coerce")
        perception_survey_df.loc[perception_survey_df[col] >= 90, col] = np.nan

perception_survey_df.to_csv(
    os.path.join(PROCESSED_DATA_PATH, "perception_survey_clean.csv"),
    index=False,
    encoding="utf-8-sig",
)

print(f"Todos los conjuntos de datos limpios han sido exportados a: {PROCESSED_DATA_PATH}")

Todos los conjuntos de datos limpios han sido exportados a: /content/drive/MyDrive/prueba/data/processed/
